In [73]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import getpass
import os 
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader # for lading the HTML content
from langchain_text_splitters import HTMLSectionSplitter # for splitting the HTML content
from dotenv import load_dotenv

load_dotenv()
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [74]:
embeddings = NVIDIAEmbeddings(
        model="nvidia/nv-embedqa-e5-v5",
        api_key=os.getenv("NVIDIA_NV_API"),
)

In [75]:
cornwall_granular_collection = Chroma(
 collection_name="cornwall_granular",
 embedding_function=embeddings,
)

# reset the collection if it already exists
cornwall_granular_collection.reset_collection()

### Loading HTML content using AsyncHTML LOADER

In [76]:
destination_url = "https://medium.com/@abhilov/whats-in-a-url-a-fun-guide-to-urls-6090fede848b"
html_loader = AsyncHtmlLoader(destination_url)
# add the documents to the collection
documents = html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.28it/s]


In [77]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)

In [ ]:
def split_docs_into_granular_chunks(documents):
    all_chunks = []
    # Use small chunks to stay under embedding token limits
    fallback_splitter = RecursiveCharacterTextSplitter(
        chunk_size=450, # here the chunk size is the size of the context window
        chunk_overlap=50, # here the chunk overlap is the size of the overlap
    )

    for doc in documents:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(html_string)
        all_chunks.extend(fallback_splitter.split_documents(temp_chunks))

    return all_chunks


In [79]:
# now split the documents into granular chunks
granular_chunks = split_docs_into_granular_chunks(documents)

#Now insert the granular chunks into the collection
cornwall_granular_collection.add_documents(granular_chunks)

['0e908933-953f-43b4-b6a1-3bfbad4b39cd',
 'b980f2f1-6ade-42cf-839d-b445c3c24e13',
 '0387fe49-a650-4c41-b483-fe4d21446d4e',
 '1ab935d3-1819-4b59-ad92-bb93078d2239',
 '92512f7f-9b93-4ef5-a377-07858d793aac',
 '20dbc8be-7926-4a86-b4be-38e9fcdb50ac',
 '4cb662d7-4f74-4455-b25f-e896de46aa7c',
 'b5a321b7-d025-4668-91a4-4cf4c7a6186f',
 '672d4fc3-7e6c-4aca-839d-1c2ba2279a5f',
 '69966c94-0872-4b29-a64b-e9d970057519',
 'ffd668e9-292f-40a2-a9c0-cad2a89069e1']

In [80]:
results = cornwall_granular_collection.similarity_search(
 query="abhilov",k=3)
for doc in results:
 print(doc.page_content)

'0',cTplC:0,cTplO:0,cTplV:5,cType: 'managed',cUPMDTk:"/@abhilov/whats-in-a-url-a-fun-guide-to-urls-6090fede848b?__cf_chl_tk=_FaJakE3QifhPNe2XHKIrf6Z8RRU0zIxWjCNyLj88w0-1777729412-1.0.1.1-F6E1BctjwaCOv0ZpGWcmMwmSoglHY9y2cGASpojl404",cvId: '3',cZone: 'medium.com',fa:"/@abhilov/whats-in-a-url-a-fun-guide-to-urls-6090fede848b?__cf_chl_f_tk=_FaJakE3QifhPNe2XHKIrf6Z8RRU0zIxWjCNyLj88w0-1777729412-1.0.1.1-F6E1BctjwaCOv0ZpGWcmMwmSoglHY9y2cGASpojl404",md:
7Nj4x1gCEn2yZLInrXybOKbffwx8jRYIiu7iHf3AwgfH1oeCLW8Uuj7vjeFG0vMovBsHNMRpw9eBwypFAeWMFF8Glmb31LoVbVF355vWKRlRIa0F8.AOdylpVn2NF33oofRwufK3SCWp113zCysW2VbhY3DXxqFfd5KkjgAXY17zqdFoNSp3C4tl1cbLzS.CZVkXk9q.7c5YySHVKJ1EfLqDF85xmd999RSz.aYxc7ZUhbtGWMoe7gzt.gt5dVC9t8uaFXj2ngJo',mdrd:
(function(){window._cf_chl_opt = {cFPWv: 'g',cH: 'NdL_oGnDfm_DGIcaQnDghMSzqYCYNU4QEqEdns7EaQM-1777729412-1.2.1.1-zAaMqkv0SnyyYnr.BS_Wl.hgW0nMCjx4jTuqGZ5s6kuEyjQmvzetPxbXJPpgZa7n',cITimeS: '1777729412',cN: 'GdHFQRR7LNIo5PICVoQJFT',cRay: '9f57749a28155790',cTplB: '0',cTplC:0,

### Embedding child chunks with ParentDocumentRetriever

In [81]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [82]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=50)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)


In [83]:
child_chunk_collection = Chroma(
 collection_name="cornwall_granular",
 embedding_function=embeddings,
)

# reset the collection if it already exists
child_chunk_collection.reset_collection()

doc_store = InMemoryStore()

In [84]:
doc_store = InMemoryStore()

In [86]:
# ParentDocumentRetriever uses:
# - parent_splitter -> splits documents into larger parent chunks
# - child_splitter -> splits parent chunks into smaller child chunks for embedding
# - vectorstore -> stores child chunks (used for similarity search)
# - docstore -> stores full parent documents (used to return full context)
parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunk_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)


In [87]:
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

base_wiki_url = "https://en.wikipedia.org/wiki/"
uk_destination_urls = [base_wiki_url + name for name in uk_destinations]


In [88]:
for item in uk_destination_urls:
    destination_url = item if str(item).startswith(("http://", "https://")) else f"https://en.wikipedia.org/wiki/{item}"
    try:
        html_loader = AsyncHtmlLoader(destination_url)
        documents = html_loader.load()
        if documents:
            parent_doc_retriever.add_documents(documents)
            print(f"Indexed: {destination_url}")
    except Exception as e:
        print(f"Skipped {destination_url}: {e}")


Fetching pages:   0%|          | 0/1 [00:00<?, ?it/s]


InvalidUrlClientError: Cornwall